In [1]:
from numpy import array, zeros, where, arange, linspace, exp, sin, cos, sort, sum, dot, ndarray, random
from numpy.random import uniform, choice, seed
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from mpl_toolkits import mplot3d

## Lets Build a Neural Network :)

Lets describe the equations! $\beta$ and $w$ are weights (bias and scalar respectively), $X = \{x_1, x_2, \ldots, x_p\}$ are the data for $p$ neurons in the previous layer, $A_{l,k}$ is the $k$th neuron in the $l$th hidden layer, $A_{l - 1, j}$ is the value of the jth neuron of the previous layer, $g(z)$ is the activation function, $f(x)$ is the output, and $N$ is the number of data points.
$$A_{l,k}(x) = g(\beta_{l,k} + \sum_{j=1}^pw_{l,k,j}A_{l - 1, j})$$

$f(x)$ is the same but with an extra bias term and sometimes without an activation function:
$$f_k(x) = \alpha_{final,k} + g(\beta_{l,k} + \sum_{j=1}^pw_{l,k,j}A_j)$$
$$f_k(x) = \beta_{l,k} + \sum_{j=1}^pw_{l,k,j}A_j$$

These functions describe the model but not how to *train* the model. Do do so, we need a loss function and backpropagation. We will be using L2 loss:
$$R(\theta) = \frac{1}{N}\sum^N_{i=1}(y - f_\theta(X_i))^2$$
$$\frac{\partial R}{\partial \theta_i} =  -2\sum^N_{i=1}(y - f_\theta(X_i))*\frac{\partial f_\theta(X_i)}{\partial \theta_i}$$
$$\frac{\partial f_\theta(X_i)}{\partial \theta_i} = \frac{\partial}{\partial \theta_i} (\alpha_{final,k} + g(\beta_{l,k} + \sum_{j=1}^pw_{l,k,j}A_j))$$



In [ ]:
class NeuralNetwork:
    '''
    This Class handles the construction, tuning, and use of a neural network.

    Intended Usage:
    model = NeuralNetwork()

    # Set the number of input neurons
    model.set_input_layer(num_neurons = 2)

    # Add hidden layer
    model.add_hidden_layer(num_neurons = 4)

    # set output layer
    model.set_output_layer(num_neurons = 1)

    # Initial prediction w/ random weights
    print(f'Model predicts: {model.forward_pass(test_data)}')

    # Train the model
    epochs = 1000
    for epoch in range(epochs):
        model.backpropagation(train_data)

    # Next prediction
    print(f'Model predicts: {model.forward_pass(test_data)}')
    '''

    def __relu(x):
      return x if x>0 else 0

    def __relu_deriv(x):
      return 1 if x>0 else 0

    # Constructor
    def __init__(self, input_neurons, def_weight_range = (-2,2)):
    # Class Properties
        self.bias_weights = list()
        self.scalar_weights = list()
        self.neuron_values = [zeros(input_neurons) - 1]
        self.deriv_values = list()
        self.act_funcs = list()
        self.final_biases = None
        self.def_weight_range = def_weight_range

    # Add Layers
    def add_layer(self, neurons, weights = None, act_func = __relu, act_func_deriv = __relu_deriv):
        num_last_layer = len(self.neuron_values[-1])
        if weights == None:
            self.bias_weights += [uniform(self.def_weight_range[0], self.def_weight_range[1], neurons)]
            self.scalar_weights += [uniform(self.def_weight_range[0], self.def_weight_range[1], (neurons, num_last_layer))]
            self.neuron_values += [zeros(neurons) - 1]
            self.deriv_values += [zeros(neurons)]
            self.act_funcs += [[act_func, act_func_deriv]]
        else:
            raise Exception("Strict assignment of weights not yet implemented")
        
    # set output layer
    def set_output_layer(self, neurons, weights = None, act_func = __relu, act_func_deriv = __relu_deriv):
        num_last_layer = len(self.neuron_values[-1])
        if weights == None:
            self.bias_weights += [uniform(self.def_weight_range[0], self.def_weight_range[1], neurons)]
            self.scalar_weights += [uniform(self.def_weight_range[0], self.def_weight_range[1], (neurons, num_last_layer))]
            self.neuron_values += [zeros(neurons) - 1]
            self.deriv_values += [zeros(neurons)]
            self.act_funcs += [[act_func, act_func_deriv]]
            self.final_biases = uniform(self.def_weight_range[0], self.def_weight_range[1], neurons)
        else: 
            raise Exception("Strict assignment of weights in layer initialization not yet implemented")

    def forward_pass(self, X):
        # Set up input
        self.neuron_values[0] = X

        # Calculate following layers
        for layer in range(1,len(self.neuron_values)):
            # List comprehension for each neuron in layer
            num_neurons = len(self.neuron_values[layer])
            new_values = array([self.calc_neuron((layer, neuron)) for neuron in range(num_neurons)])
            new_derivs = array([self.calc_neuron_deriv((layer, neuron)) for neuron in range(num_neurons)])
            self.neuron_values[layer] = new_values
            self.deriv_values[layer-1] = new_derivs
            
        self.neuron_values[-1] = self.neuron_values[-1] + self.final_biases
        return self.neuron_values[-1]

    # calculate value of neuron
    def calc_neuron(self, neuron):
        if neuron[0] == 0:
          raise Exception("Can only calculate hidden or output layers")
        layer = neuron[0]
        neuron = neuron[1]
        scalar_weights = self.scalar_weights[layer - 1][neuron]
        prev_values = self.neuron_values[layer - 1]
        act_func = self.act_funcs[layer-1][0]
        return act_func(self.bias_weights[layer - 1][neuron] + dot(prev_values, scalar_weights))

    def calc_neuron_deriv(self, neuron):
        if neuron[0] == 0:
          raise Exception("Can only calculate hidden or output layers")
        layer = neuron[0]
        neuron = neuron[1]
        scalar_weights = self.scalar_weights[layer - 1][neuron]
        prev_values = self.neuron_values[layer - 1]
        deriv = self.act_funcs[layer-1][1]
        return deriv(self.bias_weights[layer - 1][neuron] + dot(prev_values, scalar_weights))

    def __str__(self):
        print(f'Bias Weights: \n      {self.bias_weights}')
        print(f'Scalar Weights: \n      {self.scalar_weights}')
        print(f'Final Weights: \n      {self.final_biases}')
        print(f'Neuron Values: \n      {self.neuron_values}')
        print(f'Deriv Values: \n      {self.deriv_values}')
        return ""

    def set_training_data(self, x, y):
        if x.shape[1] != len(self.neuron_values[0]):
            raise Exception("Data and input layer have different dimensions!")
        y_output_size = 1 if len(y.shape) == 1 or y.shape[1] == 1 else y.shape[1]
        if y_output_size != len(self.neuron_values[-1]):
            raise Exception("Data and output layer have different dimensions!")
        if x.shape[0] != y.shape[0]:
            raise Exception("X and Y must be equal!")
        self.x_training = x
        self.y_training = y

    def backpropagation(self, learning_rate=0.001, batch=20, seed=42):
        num_layers = len(self.neuron_values)
        neurons_per_layer = [len(x) for x in self.neuron_values]
        LR = learning_rate / batch

        random.seed(seed)

        x_batch = self.x_training[choice(len(self.x_training), batch)]
        y_batch = self.y_training[choice(len(self.y_training), batch)]

        for X,Y in zip(x_batch, y_batch):
            
            # Calculate Neuron values and derivatives
            y_pred = self.forward_pass(X)

            # Get the difference between predicted and observed values
            R = (y_pred - Y) / batch
            
            # Final Biases
            self.final_biases += -1 * LR * R

            # Final Layer inner Biases
            self.bias_weights[-1] += -1 * LR * R * self.deriv_values[-1]

            # Scalar Weights
            if neurons_per_layer[-1] == 1:
                self.scalar_weights[-1] += -1 * LR * R * self.deriv_values[-1] * self.neuron_values[-2]
            else:
                # TODO following list comprehension is suspect
                self.scalar_weights[-1] += -1 * LR * R * array([self.deriv_values[-1][j] * self.neuron_values[-2] for j in range(neurons_per_layer[-1])])

            # Currently partial f / partial f hence an array of ones...
            pfpLast = zeros(neurons_per_layer[-1]) + 1

            # optimize hidden layer weights
            for layer in range(num_layers - 2, 0, -1):
                
                # partial DS layer / partial current layer Calculation - calculated for ea neuron in current layer
                pLastpAs = [self.deriv_values[layer] * self.scalar_weights[layer][:,neuron] for neuron in range(neurons_per_layer[layer])]

                # partial f(x) / partial current layer - calculated using dot(partial f / partial last, partial last / partial A)
                pfpAs = [dot(pfpLast, pLastpA) for pLastpA in pLastpAs]

                # update partial f(x) / partial US layer to match partial f(x) / partial current layer for next loop
                pfpLast = pfpAs

                # bias weights
                self.bias_weights[layer - 1] += -1 * LR * R * pfpAs * self.deriv_values[layer - 1]

                # scalar weights
                pfpOmegas = array([pfpAs[k] * self.deriv_values[layer-1][k] * self.neuron_values[layer-1] for k in range(neurons_per_layer[layer])])
                self.scalar_weights[layer - 1] += -1 * LR * R * pfpOmegas
    
    def backpropagation2(self, learning_rate=0.001, batch=20, seed=42):
        num_layers = len(self.neuron_values)
        neurons_per_layer = [len(x) for x in self.neuron_values]
        LR = learning_rate / batch

        random.seed(seed)

        x_batch = self.x_training[choice(len(self.x_training), batch)]
        y_batch = self.y_training[choice(len(self.y_training), batch)]

        # Weight Buffers
        bias_weight_b = [zeros(neurons_per_layer[layer]) for layer in range(1, num_layers - 1)]
        scalar_weight_b = [zeros((neurons_per_layer[layer], neurons_per_layer[layer-1])) for layer in range(1, num_layers - 1)]
        final_weight_b = zeros(neurons_per_layer[-1])


        for X,Y in zip(x_batch, y_batch):
            
            # Calculate Neuron values and derivatives
            y_pred = self.forward_pass(X)

            # Get the difference between predicted and observed values
            R = (y_pred - Y) / batch
            
            # Final Biases
            final_weight_b += -1 * LR * R

            # Final Layer inner Biases
            bias_weight_b[-1] += -1 * LR * R * self.deriv_values[-1]

            # Scalar Weights
            if neurons_per_layer[-1] == 1:
                scalar_weight_b[-1] += -1 * LR * R * self.deriv_values[-1] * self.neuron_values[-2]
            else:
                # TODO following list comprehension is suspect
                scalar_weight_b[-1] += -1 * LR * R * array([self.deriv_values[-1][j] * self.neuron_values[-2] for j in range(neurons_per_layer[-1])])

            # Currently partial f / partial f hence an array of ones...
            pfpLast = zeros(neurons_per_layer[-1]) + 1

            # optimize hidden layer weights
            for layer in range(num_layers - 2, 0, -1):
                
                # partial DS layer / partial current layer Calculation - calculated for ea neuron in current layer
                pLastpAs = [self.deriv_values[layer] * self.scalar_weights[layer][:,neuron] for neuron in range(neurons_per_layer[layer])]

                # partial f(x) / partial current layer - calculated using dot(partial f / partial last, partial last / partial A)
                pfpAs = [dot(pfpLast, pLastpA) for pLastpA in pLastpAs]

                # update partial f(x) / partial US layer to match partial f(x) / partial current layer for next loop
                pfpLast = pfpAs

                # bias weights
                bias_weight_b[layer - 1] += -1 * LR * R * pfpAs * self.deriv_values[layer - 1]

                # scalar weights
                pfpOmegas = array([pfpAs[k] * self.deriv_values[layer-1][k] * self.neuron_values[layer-1] for k in range(neurons_per_layer[layer])])
                scalar_weight_b[layer - 1] += -1 * LR * R * pfpOmegas
        
        for layer in range(num_layers-1):
            self.bias_weights[layer] = bias_weight_b[layer]
            
    
    def set_scalar_weights(self, layer, weights):
        weights = weights if isinstance(weights, ndarray) else array(weights)
        self.scalar_weights[layer - 1] = weights.astype(float)
    
    def set_bias_weights(self, layer, weights):
        weights = weights if isinstance(weights, ndarray) else array(weights)
        self.bias_weights[layer - 1] = weights.astype(float)
    
    def set_final_biases(self, weights):
        weights = weights if isinstance(weights, ndarray) else array(weights)
        self.final_biases = weights.astype(float)

    def save_weights(self, filepath):
        
        layers = len(self.neuron_values)

        # Final Weights
        file = "-Final Weights-\n"
        file += f'  [{",".join([x for x in self.final_biases.astype(str)])}]\n'

        # Bias Weights
        file += "-Bias Weights-\n"
        for layer in range(1, layers):
            file += f'  [{",".join([x for x in self.bias_weights[layer-1].astype(str)])}]\n'

        # scalar weights
        file += "-Scalar Weights-\n"
        for layer in range(1, layers):
            file += '  [\n'
            temp = ""
            for neuron in range(len(self.neuron_values[layer])):
                temp += f'    [{",".join([x for x in self.scalar_weights[layer-1][neuron].astype(str)])}],'
                temp += "" if neuron == len(self.neuron_values[layer]) - 1 else "\n"
            file += f'{temp}\n  ],\n'

        with open(filepath, 'w') as f:
            f.write(file)

    def load_weights(self, filepath):
        from re import compile
        re_values = compile('(-?\\d+\\.?\\d*),?')
        re_new_layer = compile('\\s+],')

        with open(filepath, 'r') as f:

            f.readline()
            line = f.readline()
            while "bias" not in line.lower():
                self.final_biases = array(re_values.findall(line), dtype=float)
                line = f.readline()
            
            line = f.readline()
            layer = 0
            while "scalar" not in line.lower():
                self.bias_weights[layer] = array(re_values.findall(line), dtype=float)
                layer += 1
                line = f.readline()
            
            f.readline()
            layer = 0
            neuron = 0
            line = f.readline()
            while line != "":
                if re_new_layer.match(line) != None:
                    neuron = 0
                    layer += 1
                    f.readline()
                else:
                    self.scalar_weights[layer][neuron] = array(re_values.findall(line), dtype=float)
                    neuron += 1
                line = f.readline()

    def rmse(self, batch = 10, seed=42):
        random.seed(seed)

        x_batch = self.x_training[choice(len(self.x_training), batch)]
        y_batch = self.y_training[choice(len(self.y_training), batch)]

        return (sum([(Y - self.forward_pass(X))**2 for X,Y in zip(x_batch, y_batch)], axis=0) / batch)**(1/2)


## Test Neural Network Framework

In [389]:
sigmoid = lambda x: 1 / (1 + exp(-x))
sigmoid_deriv = lambda x: sigmoid(x)*(1 - sigmoid(x))

In [390]:
def act_func(x):
    return sin(x) / x
def act_func_deriv(x):
    return (cos(x) * x - sin(x)) / x**2

In [391]:

x = uniform(-10, 10, 5000)
y = uniform(-10, 10, 5000)

f = lambda x,y: x**2 + y**2

z = [f(x,y) + uniform(-0.1, 0.1) for x,y in zip(x,y)]

In [392]:
new_NN = NeuralNetwork(2, def_weight_range=(-5, 5))
new_NN.add_layer(3, act_func=act_func, act_func_deriv=act_func_deriv)
new_NN.add_layer(2, act_func=act_func, act_func_deriv=act_func_deriv)
new_NN.set_output_layer(1)
new_NN.set_training_data(x=array([x[:500],y[:500]]).T, y=array(z)[:500])

In [370]:
init_rmse = new_NN.rmse(batch=50)
print(f'initial RMSE: {init_rmse}')
LR = 0.5
epochseed = uniform(0, 100000, 100).astype(int)
for seed in epochseed:
    new_NN.backpropagation(learning_rate=LR, batch=20, seed=seed)
    LR = LR * 0.95
new_rmse = new_NN.rmse(batch=50)
print(f'RMSE diff: {new_rmse - init_rmse}, new RMSE: {new_rmse}')

initial RMSE: [77.55452299]
RMSE diff: [-24.94183928], new RMSE: [52.6126837]


In [366]:
new_NN.load_weights('100_epochs')
new_NN.forward_pass([1,1])

array([46.27673202])

In [367]:
new_NN.load_weights('0_epochs')
new_NN.forward_pass([1,1])

array([12.10843734])

In [358]:
f(1,1)

2

In [349]:
new_NN.forward_pass([1,1])

array([23.04468141])

In [369]:
new_NN.save_weights('0_epochs')

In [371]:
new_NN.save_weights('100_epochs')

## Handwriting Data!